# 03 – Grover search (3 qubits)

In [1]:
# Imports and setup: we search 8 items for the marked state |101>
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()
n, marked = 3, "101"

In [2]:
# Oracle: flips the phase of the marked state only. X gates turn the '101' pattern into '111',
# a multi-controlled Z flips the sign of |111>, then X gates undo the change.
def oracle():
    qc = QuantumCircuit(n)
    for i, b in enumerate(reversed(marked)):   # Qiskit bit order: qubit 0 is rightmost
        if b == "0": qc.x(i)
    qc.h(n-1); qc.mcx(list(range(n-1)), n-1); qc.h(n-1)  # CCZ
    for i, b in enumerate(reversed(marked)):
        if b == "0": qc.x(i)
    return qc

In [3]:
# Diffuser: reflects amplitudes about their mean, which amplifies the marked state
def diffuser():
    qc = QuantumCircuit(n)
    qc.h(range(n)); qc.x(range(n))
    qc.h(n-1); qc.mcx(list(range(n-1)), n-1); qc.h(n-1)
    qc.x(range(n)); qc.h(range(n))
    return qc

In [4]:
# Optimal number of iterations ~ (pi/4)*sqrt(N); for N=8 that is 2
iters = int(np.floor(np.pi / 4 * np.sqrt(2**n)))
qc = QuantumCircuit(n, n)
qc.h(range(n))                    # uniform superposition over all 8 states
for _ in range(iters):
    qc.compose(oracle(), inplace=True)
    qc.compose(diffuser(), inplace=True)
qc.measure(range(n), range(n))
print("iterations:", iters)

iterations: 2


In [5]:
# Run: the marked state should show up ~94% of the time (vs 12.5% by random guessing)
counts = sim.run(transpile(qc, sim), shots=2000).result().get_counts()
print(counts)
assert max(counts, key=counts.get) == marked

{'111': 16, '011': 15, '001': 15, '100': 22, '110': 26, '010': 16, '000': 17, '101': 1873}
